In [1]:
import refinitiv.data as rd
import pandas as pd
import numpy as np


# Default settings markets

In [7]:
%load_ext autoreload
%autoreload 2

In [ ]:
markets = ["XPAR", "XAMS", "XBRU", "XMAD", "XNYS", "XNAS"]
ESG_SCORE_THRESH_HOLD = 50
MARKET_VALUE_THRESH_HOLD = 1e9

# STOCK SCREENER 

### ESG Score Screening and Market Value Screening

In [3]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x11d93ca40 {name='workspace'}>

In [6]:
mic_codes = ", ".join(markets)
screener_query = (
    "SCREEN("
    "U(IN(Equity(active,public,primary))), "
    f"IN(TR.ExchangeMarketIdCode, {mic_codes}), "
    "TR.TRESGScore > 50"
    ")"
)

In [14]:
df_screener = rd.get_data(screener_query,
                          fields =[
                              "TR.CommonName",
                              "TR.TRESGScore",
                              "TR.CompanyMarketCap(CURN=USD)"
                          ]
)

In [ ]:
df_screener = df_screener[df_screener["Company Market Cap"] >= MARKET_VALUE_THRESH_HOLD]
df_screener["Company Market Cap"].min()


50.0404235126541

In [39]:
df_screener.head()

,Instrument,Company Common Name,ESG Score,Company Market Cap
0,FN.N,Fabrinet,58.673081,20989447166.790001
1,TGLS.N,Tecnoglass Inc,70.471181,2326609522.16
3,AMA.MC,Amadeus IT Group SA,89.508399,26132188596.786499
4,ZWS.N,Zurn Elkay Water Solutions Corp,69.358455,8441645464.14
5,EDEN.PA,Edenred SE,79.999928,5121362302.25413


# Fetch Data for the selected stocks

### Settings for historical data retrieval

In [69]:
selected_stocks = df_screener.sort_values(by="Company Market Cap", ascending=False)["Instrument"].tolist()
selected_stocks_by_name = df_screener[["Company Common Name", "Instrument"]]
pd.DataFrame(selected_stocks).to_csv("selected_stocks.csv", index=False)
selected_stocks[:5]


['LLY.N', 'JPM.N', 'XOM.N', 'JNJ.N', 'ASML.AS']

In [75]:
# Rolling window of 3 months
rolling_window = 4 * 21  # Assuming 21 trading days in a month

In [76]:
END_DATE = pd.Timestamp.today()
START_DATE = END_DATE - pd.Timedelta(days=rolling_window)


### Fetch close price history for the selected stocks

In [77]:
df_history = rd.get_history(
    universe=selected_stocks[:300],
    interval='1d',
    fields=['TRDPRC_1'],
    start=START_DATE,
    end=END_DATE
)

/opt/anaconda3/envs/final_project/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


### Upload the Start and End date to the database

In [81]:
df_history.shape
df_history.head()

TRDPRC_1,LLY.N,JPM.N,XOM.N,JNJ.N,ASML.AS,V.N,MA.N,ABBV.N,PG.N,HD.N,...,BOUY.PA,IP.N,VLTO.N,AMCR.N,DGX.N,PERP.PA,KEY.N,RL.N,PSTG.N,TSN.N
Date,,,,,,,,,,,,,,,,,,,,,
2025-12-04,1014.49,316.1,117.14,202.48,957.3,327.1,542.31,228.71,145.36,351.17,...,43.37,39.13,102.89,41.6,184.18,76.7,19.11,356.97,72.2,56.14
2025-12-05,1010.31,315.04,116.54,201.93,951.6,331.24,545.52,226.08,143.45,354.61,...,43.17,39.06,102.16,41.5,182.51,76.88,19.26,368.42,70.43,56.92
2025-12-08,997.59,315.21,115.98,201.62,963.2,326.84,540.44,223.12,138.34,349.91,...,43.71,38.51,99.62,41.25,181.82,75.22,19.39,356.44,71.04,56.22
2025-12-09,982.22,300.51,118.25,199.96,952.9,326.5,537.55,222.99,139.63,345.27,...,43.7,37.58,98.41,40.55,179.62,73.7,19.98,355.53,70.15,55.91
2025-12-10,993.64,310.11,119.54,206.54,946.0,325.73,538.86,225.18,139.82,351.13,...,43.13,39.12,97.75,41.0,179.51,73.32,20.52,357.7,73.65,57.67


# Analyst the stock data

### Change in returns 

In [82]:
df_returns_change = df_history.columns.pct_change().dropna()

AttributeError: 'Index' object has no attribute 'pct_change'